# IE643 - Teacher caching (CheXagent-2-3b), sharded across sessions

**Accelerator:** GPU T4 x2 (one worker per GPU; a single GPU also works). **Internet:** on.

**Inputs to attach**
1. Notebook output **IE643 Cleaning Step 7** (final `reports_clean.csv`) and notebook output **IE643 Cleaning** (`images_512.npy`). Both are needed.
2. From the 2nd session on: **this notebook's own previous output** (Add Input -> Your Work -> this notebook). Its `teacher_cache/` folder is copied in and the run resumes from there.

**Workflow**
1. `MODE = "probe"` once (~20-30 min of GPU). Read the report at the bottom, then set `ROUTE`, `BATCH_SIZE`, `CACHE_TOPK`.
2. `MODE = "cache"` with **Save & Run All (commit)**. Workers stop on their own before the 12 h limit and everything done so far is saved in the output. Repeat (with the previous output attached) until the summary says 0 remaining.
3. `MODE = "merge"` (CPU session is enough) to write the final aligned files to `teacher_final/`.

Do not change `ROUTE`, `CACHE_FEATURES` or `CACHE_TOPK` between cache sessions: the notebook refuses to mix them.

In [ ]:
import time
T0 = time.time()                      # session start; the worker deadline is measured from here

NOTEBOOK_VERSION = "v3-mplbackend-fix"   # if your run does not print this line, you are running an OLD copy
print("notebook version:", NOTEBOOK_VERSION)
MODE = "smoke"                        # "smoke" (CPU session, Accelerator None, FREE) -> "probe" (GPU) -> "cache" (GPU, repeat) -> "merge" (CPU)
SPLITS = ["val", "train"]             # val first so a teacher-vs-GT check is possible early; add "test" only at the end
ROUTE = "two_step"                    # from the probe: "two_step" (official findings -> summary) or "direct"
USE_INDICATION = True                 # used only if an indication/history column exists in reports_clean.csv
CACHE_FEATURES = True                 # pooled SigLIP + projected visual features for feature distillation
CACHE_TOPK = False                    # True only if the CPU tokenizer check says the token IDs match
TOPK = 20
BATCH_SIZE = 4                        # from the probe's throughput table
CHUNK = 64                            # images per saved chunk (the most work lost if a session is killed)
MAX_NEW_FINDINGS = 256
MAX_NEW_IMPRESSION = 128
SESSION_BUDGET_H = 11.0               # Kaggle kills at 12 h; workers stop cleanly before T0 + this
MAX_IMAGES_THIS_SESSION = None        # e.g. 3000 for a deliberately short session
PROBE_N = 24

CSV_PATH = None                       # None = use the known Step 7 output (see resolve_data); set explicitly to override
NPY_PATH = None
USE_PRIVATE_ENV = True                # build a Python 3.10 env for CheXagent (needed on Kaggle's Python 3.13 image)
EMPTY_INDICATION_TEXT = ""            # reports_clean.csv has no clinical-history column, so the findings prompt gets this text
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
TMP_DIR = "/tmp"
N_GPU = None                          # None = all visible GPUs
DEVICE = "cuda:0"                     # per worker (each worker only sees its own GPU)
if MODE == "smoke":
    DEVICE = "cpu"                    # the smoke test deliberately runs on CPU so it costs no GPU quota

In [ ]:
import os, sys, json, glob, shutil, subprocess
import numpy as np, pandas as pd

DEPS_SCRIPT = r"""
import ast, sys, json, glob, os, importlib.util
d, only = sys.argv[1], (sys.argv[2] if len(sys.argv) > 2 else "")
std, files = set(sys.stdlib_module_names), sorted(glob.glob(d + "/*.py"))
local = {os.path.basename(f)[:-3] for f in files}
req, opt = {}, {}
for f in files:
    if only and os.path.basename(f) not in only.split(","):
        continue
    tree = ast.parse(open(f, encoding="utf-8").read())
    def visit(node, nested):
        for ch in ast.iter_child_nodes(node):
            names = [a.name for a in ch.names] if isinstance(ch, ast.Import) else \
                    ([ch.module] if isinstance(ch, ast.ImportFrom) and ch.level == 0 and ch.module else [])
            for n in names:
                top = n.split(".")[0]
                if top not in std and top not in local:
                    (opt if nested else req).setdefault(top, []).append(os.path.basename(f))
            visit(ch, nested or isinstance(ch, (ast.Try, ast.If, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.With)))
    visit(tree, False)
def missing(mods):
    out = []
    for m in mods:
        try:
            ok = importlib.util.find_spec(m) is not None
        except Exception:
            ok = False
        if not ok:
            out.append(m)
    return sorted(out)
print(json.dumps({"missing_required": missing(req), "required": sorted(req), "optional_missing": missing(opt)}))
"""
PIP_NAME = {"cv2": "opencv-python-headless", "PIL": "pillow", "yaml": "pyyaml", "sklearn": "scikit-learn", "skimage": "scikit-image", "bs4": "beautifulsoup4"}

def resolve_deps(py, code_dir, only="", constraints=("numpy<2",)):
    """Read every .py in code_dir, find the packages it imports at module level, install what the env is missing."""
    env = dict(os.environ, MPLBACKEND="agg", UV_CACHE_DIR="/tmp/uvcache")
    for attempt in range(3):
        r = subprocess.run([py, "-c", DEPS_SCRIPT, code_dir, only], capture_output=True, text=True, env=env)
        assert r.returncode == 0, "dependency scan failed: " + r.stderr[-1500:]
        info = json.loads(r.stdout.strip().splitlines()[-1])
        if not info["missing_required"]:
            print(f"remote-code imports OK (required: {info['required']}); conditional/optional and not installed: {info['optional_missing']}")
            return info
        pk = [PIP_NAME.get(m, m) for m in info["missing_required"]]
        print("installing packages the model code imports:", pk)
        r = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", py] + pk + list(constraints), capture_output=True, text=True, env=env)
        assert r.returncode == 0, "could not install " + str(pk) + ": " + r.stderr[-1500:]
    raise RuntimeError(f"still missing after installing: {info['missing_required']}")

def fetch_remote_code(py, repo, code_dir):
    """Only the small code/config files of a HF repo (no weights)."""
    r = subprocess.run([py, "-c", "import sys; from huggingface_hub import snapshot_download as s; s(sys.argv[1], allow_patterns=['*.py', '*.json', '*.txt'], local_dir=sys.argv[2])",
                        repo, code_dir], capture_output=True, text=True, env=dict(os.environ, MPLBACKEND="agg"))
    assert r.returncode == 0, f"could not fetch code files of {repo}: " + r.stderr[-1500:]

def setup_private_env(pkgs, check_code, venv, pythons=("3.10", "3.11")):
    """Kaggle's default Python is 3.13; transformers==4.40.0 needs tokenizers 0.19 which has no 3.13 wheel.
    So CheXagent-related code runs in its own Python 3.10 environment (what the CheXagent README pins), built with uv."""
    py = f"{venv}/bin/python"
    def ok():
        return os.path.exists(py) and subprocess.run([py, "-c", check_code], capture_output=True, env=dict(os.environ, MPLBACKEND="agg")).returncode == 0
    if ok():
        return py
    t = time.time()
    env = dict(os.environ, MPLBACKEND="agg", UV_CACHE_DIR="/tmp/uvcache")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    if not os.path.exists(py):
        for v in pythons:
            r = subprocess.run([sys.executable, "-m", "uv", "venv", "--python", v, venv], capture_output=True, text=True, env=env)
            if r.returncode == 0:
                break
            print(f"uv venv python {v} failed:", r.stderr[-400:])
        else:
            raise RuntimeError("could not create a private Python environment")
    r = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", py] + list(pkgs), capture_output=True, text=True, env=env)
    print(r.stdout[-1500:], r.stderr[-3000:])
    assert r.returncode == 0, "package install into the private environment failed (log above)"
    assert ok(), f"private environment check failed: {check_code}"
    print(f"private environment ready at {venv} in {time.time() - t:.0f}s")
    return py

CACHE = f"{WORK}/teacher_cache"
LOGS = f"{WORK}/logs"
os.makedirs(CACHE, exist_ok=True)
os.makedirs(LOGS, exist_ok=True)

# ---- data: CSV from "IE643 Cleaning Step 7", images from "IE643 Cleaning"
KNOWN_CSV = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning-step-7/reports_clean.csv"   # FINAL csv (Step 7 output)
KNOWN_NPY = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning/images_512.npy"            # images live in the ORIGINAL cleaning output

def resolve_data():
    csv, npy = CSV_PATH, NPY_PATH
    if csv is None and os.path.exists(KNOWN_CSV):
        csv = KNOWN_CSV
    if npy is None and os.path.exists(KNOWN_NPY):
        npy = KNOWN_NPY
    if csv is None:                                   # fallback: any reports_clean.csv that has the Step 7 columns
        c = [p for p in glob.glob(f"{INPUT_ROOT}/**/reports_clean.csv", recursive=True)
             if {"impression_final", "use_for_training"} <= set(pd.read_csv(p, nrows=0).columns)]
        assert len(c) == 1, f"Need exactly one reports_clean.csv containing impression_final; found {c}. Attach the 'IE643 Cleaning Step 7' notebook output."
        csv = c[0]
    if npy is None:
        c = glob.glob(f"{INPUT_ROOT}/**/images_512.npy", recursive=True)
        assert len(c) == 1, f"Need exactly one images_512.npy; found {c}. Attach the 'IE643 Cleaning' notebook output."
        npy = c[0]
    assert os.path.exists(csv) and os.path.exists(npy), f"missing {csv} or {npy}; attached inputs: {os.listdir(INPUT_ROOT)}"
    return csv, npy

CSV, NPY = resolve_data()
df = pd.read_csv(CSV, low_memory=False)
imgs = np.load(NPY, mmap_mode="r")
need = ["idx", "subset", "impression_final", "use_for_training", "y_no_finding"]
missing = [c for c in need if c not in df.columns]
assert not missing, f"reports_clean.csv is missing {missing} (did the Step 7 cleaning run finish?)"
assert len(df) == imgs.shape[0] and (df["idx"].values == np.arange(len(df))).all(), "CSV and images_512.npy are not aligned"
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
assert imgs.shape[1:] == (512, 512) and imgs.dtype == np.uint8, f"unexpected image array {imgs.shape} {imgs.dtype}"
print(f"csv: {CSV}\nnpy: {NPY}\n  rows {len(df)} | images {imgs.shape} {imgs.dtype}")
print("  usable per split:", df[df.use_for_training].subset.value_counts().to_dict())

# ---- indication column (CheXagent's findings prompt takes it; the student never sees it)
IND_CANDS = ["indication", "section_indication", "section_clinical_history", "section_history", "clinical_history", "history"]
IND_COL = None
for c in IND_CANDS:
    if c in df.columns:
        cov = df[c].fillna("").astype(str).str.strip().ne("").mean()
        print(f"  column {c!r}: non-empty in {cov:.1%} of rows")
        if IND_COL is None and cov > 0.2:
            IND_COL = c
other = [c for c in df.columns if ("indic" in c.lower() or "history" in c.lower()) and c not in IND_CANDS]
if other:
    print("  other possible indication columns (not used):", other)
if not USE_INDICATION:
    IND_COL = None
print("indication column used:", IND_COL, "" if IND_COL else f"(none in the CSV -> findings prompt uses indication text {EMPTY_INDICATION_TEXT!r})")

# ---- GPUs
if MODE in ("smoke", "probe", "cache"):
    import torch
    if MODE == "smoke" and torch.cuda.device_count() > 0:
        print("WARNING: a GPU is attached. The smoke test runs on CPU and would waste GPU quota - set Accelerator to None.")
    n_vis = torch.cuda.device_count() if DEVICE.startswith("cuda") else 1
    N_GPU = n_vis if N_GPU is None else N_GPU
    assert N_GPU >= 1, "No GPU visible: set Accelerator to GPU T4 x2"
    if DEVICE.startswith("cuda"):
        print("GPUs:", [torch.cuda.get_device_name(i) for i in range(n_vis)], "| using", N_GPU)
    # CheXagent needs transformers==4.40.0 (its code asserts this) -> workers run in a private Python 3.10 environment
    PY = sys.executable
    if USE_PRIVATE_ENV:
        PY = setup_private_env(
            ["torch==2.7.1", "torchvision==0.22.1", "transformers==4.40.0", "einops", "albumentations", "opencv-python-headless",
             "accelerate", "sentencepiece", "protobuf", "safetensors", "pandas", "pillow", "numpy<2", "requests", "matplotlib", "pyarrow"],
            "import torch, transformers, einops, albumentations, cv2, pandas, PIL; assert transformers.__version__ == '4.40.0'",
            venv="/tmp/cxenv")
    if USE_PRIVATE_ENV:                    # read ALL of CheXagent's code files and install anything they import that is missing
        fetch_remote_code(PY, "StanfordAIMI/CheXagent-2-3b", "/tmp/cx_code")
        resolve_deps(PY, "/tmp/cx_code")
    v = subprocess.run([PY, "-c", "import sys, torch, transformers; print(sys.version.split()[0], torch.__version__, transformers.__version__, "
                        "torch.cuda.is_available(), torch.cuda.device_count())"], capture_output=True, text=True, env=dict(os.environ, MPLBACKEND="agg"))
    print("worker env (python, torch, transformers, cuda, n_gpu):", v.stdout.strip(), v.stderr.strip()[-300:])
    assert v.stdout.split()[2] == "4.40.0", "transformers 4.40.0 is not what the workers will use"
    if DEVICE.startswith("cuda"):
        assert v.stdout.split()[3] == "True", "the worker environment's torch cannot see the GPU"
        assert int(v.stdout.split()[4]) >= N_GPU, "worker environment sees fewer GPUs than expected"

# ---- bring in previous sessions' cache (attached as input), without overwriting anything here
def import_previous_cache():
    n_npz = n_lines = 0
    for d in sorted(glob.glob(f"{INPUT_ROOT}/**/teacher_cache", recursive=True)):
        if os.path.realpath(d) == os.path.realpath(CACHE):
            continue
        for f in sorted(os.listdir(d)):
            src, dst = f"{d}/{f}", f"{CACHE}/{f}"
            if f.endswith(".npz") and not f.endswith(".tmp.npz") and not os.path.exists(dst):
                shutil.copy2(src, dst); n_npz += 1
            elif f.endswith(".jsonl"):
                have = set(open(dst).read().splitlines()) if os.path.exists(dst) else set()
                new = [l for l in open(src).read().splitlines() if l.strip() and l not in have]
                with open(dst, "a") as fh:
                    fh.writelines(l + "\n" for l in new)
                n_lines += len(new)
    print(f"previous cache imported: {n_npz} chunk files, {n_lines} jsonl lines")

def load_done():
    """idx -> record, only for records whose chunk .npz exists (a crash between the two writes never counts)."""
    done = {}
    for f in sorted(glob.glob(f"{CACHE}/g*.jsonl")):
        if f.endswith("_failed.jsonl"):
            continue
        for line in open(f):
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue                       # a half-written last line from a killed session
            if os.path.exists(f"{CACHE}/{r['chunk']}.npz") and r["idx"] not in done:
                done[r["idx"]] = r
    return done

import_previous_cache()
done = load_done()
print(f"cached so far: {len(done)}", pd.Series([r["split"] for r in done.values()]).value_counts().to_dict() if done else "")

In [ ]:
%%writefile /kaggle/working/teacher_worker.py
# teacher_worker.py  -  runs CheXagent-2-3b on ONE GPU (written out by the IE643 teacher-cache notebook).
# Modes:
#   probe : quality / speed / correctness checks on ~24 val images, writes probe_report.json + probe_samples.csv
#   cache : runs the chosen route over a list of idx, writes chunked outputs that a later session can resume from
import os, sys, json, time, math, re, argparse, traceback, gc
os.environ["MPLBACKEND"] = "agg"   # Jupyter's inline backend is not installable in the private env; must be set before any import

import numpy as np
import pandas as pd

MODEL = "StanfordAIMI/CheXagent-2-3b"
SYSTEM = "You are a helpful assistant."
# Prompts copied from Stanford-AIMI/CheXagent model_chexagent/chexagent.py (findings_generation, findings_summarization)
PROMPT_FINDINGS = 'Given the indication: "{ind}", write a structured findings section for the CXR.'
PROMPT_SUMMARIZE = "Summarize the following Findings: {findings}"
# Prompt used by the old Layer-2 notebook (not one of CheXagent's trained task phrasings) - compared in the probe
PROMPT_DIRECT = "Provide the IMPRESSION section of the radiology report for this chest X-ray."

ap = argparse.ArgumentParser()
ap.add_argument("--rank", type=int, required=True)
ap.add_argument("--mode", choices=["smoke", "probe", "cache"], required=True)
ap.add_argument("--cfg", required=True)          # json config written by the notebook
ap.add_argument("--ids", default=None)            # json list of idx (cache mode)
args = ap.parse_args()
RANK = args.rank
CFG = json.load(open(args.cfg))
DEVICE = CFG.get("device", "cuda:0")


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] [gpu{RANK}] {msg}", flush=True)


import torch
import torch.nn.functional as F
from PIL import Image

DT = getattr(torch, CFG.get("dtype", "float16"))


def sync():
    if DEVICE.startswith("cuda"):
        torch.cuda.synchronize()


def is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()


# ------------------------------------------------------------------ data
df = pd.read_csv(CFG["csv"], low_memory=False)
imgs = np.load(CFG["npy"], mmap_mode="r")
assert len(df) == imgs.shape[0], (len(df), imgs.shape)
assert (df["idx"].values == np.arange(len(df))).all(), "idx column must be 0..N-1 in order"
df["impression_final"] = df["impression_final"].fillna("").astype(str)
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
IND_COL = CFG.get("indication_col")
TMP = CFG.get("tmp_dir", "/tmp") + f"/cxr_g{RANK}"
os.makedirs(TMP, exist_ok=True)


def indication(i):
    if not IND_COL:
        return ""
    v = df.at[i, IND_COL]
    if not isinstance(v, str):
        return ""
    return re.sub(r"\s+", " ", v).strip().replace('"', "'")[: CFG.get("indication_max_chars", 300)]


def image_path(i):
    p = f"{TMP}/{i}.png"
    if not os.path.exists(p):
        a = np.asarray(imgs[i])
        if a.ndim == 3 and a.shape[-1] in (1, 3):
            a = a[..., 0]
        Image.fromarray(a.astype(np.uint8)).save(p)   # lossless; the model's own loader converts to RGB and resizes
    assert len(p) < 1000, "image path longer than CheXagent's 1024-token image span"
    return p


# ------------------------------------------------------------------ teacher
class Teacher:
    def __init__(self):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        t0 = time.time()
        self.tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        dm = {"": 0} if DEVICE.startswith("cuda") else None
        try:
            m = AutoModelForCausalLM.from_pretrained(MODEL, trust_remote_code=True, torch_dtype=DT, device_map=dm)
        except Exception as e:
            log(f"fp16 load failed ({e!r}); retrying the official way (fp32 load, then cast)")
            gc.collect()
            m = AutoModelForCausalLM.from_pretrained(MODEL, trust_remote_code=True, device_map=dm)
        self.model = m.to(DT).eval()
        if not DEVICE.startswith("cuda"):
            self.model = self.model.to(DEVICE)
        self.load_s = time.time() - t0
        census = {}
        for p in self.model.parameters():
            census[str(p.dtype)] = census.get(str(p.dtype), 0) + p.numel()
        self.dtype_census = census
        self.eos = self.tok.eos_token_id
        assert self.eos is not None, "tokenizer has no eos token"
        self.vocab = self.model.get_output_embeddings().weight.shape[0]
        # ---- feature capture: wrap visual.encode (called once per generate, on the first step) + hook SigLIP tower
        self.visual = self.model.model.visual
        self.capturing = False
        self.cap = {}
        orig_encode = self.visual.encode

        def encode(image_paths, training=False):
            out = orig_encode(image_paths, training)
            if self.capturing:
                self.cap["proj_mean"] = out.detach().float().mean(1).half().cpu().numpy()
                self.cap["n_encode_calls"] = self.cap.get("n_encode_calls", 0) + 1
            return out

        self.visual.encode = encode

        def hook(mod, inp, out):
            if not self.capturing:
                return
            hs = out.hidden_states[-1]                    # what CheXagent itself feeds forward
            h = hs.detach().float()
            B, N, C = h.shape
            g = int(round(math.sqrt(N)))
            assert g * g == N, f"SigLIP token count {N} is not a square grid"
            self.cap["siglip_mean"] = h.mean(1).half().cpu().numpy()                        # [B, C]
            grid = h.reshape(B, g, g, C).permute(0, 3, 1, 2)
            grid = F.adaptive_avg_pool2d(grid, 4).permute(0, 2, 3, 1).reshape(B, 16, C)       # [B, 16, C]
            self.cap["siglip_grid4"] = grid.half().cpu().numpy()
            self.cap["n_tokens"] = N

        self.visual.model.register_forward_hook(hook)
        log(f"model loaded in {self.load_s:.0f}s | dtypes {census} | vocab {self.vocab} | eos {self.eos}")
        if DEVICE.startswith("cuda"):
            log(f"GPU memory after load: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

    # ---- prompt building, mirroring CheXagent.generate() in the official repo
    def conv(self, path, prompt):
        items = ([{"image": path}] if path else []) + [{"text": prompt}]
        q = self.tok.from_list_format(items)
        return [{"from": "system", "value": SYSTEM}, {"from": "human", "value": q}]

    def ids(self, path, prompt):
        return self.tok.apply_chat_template(self.conv(path, prompt), add_generation_prompt=True, return_tensors="pt")[0].tolist()

    def pad_left(self, seqs):
        L = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), L), self.eos, dtype=torch.long)
        mask = torch.zeros((len(seqs), L), dtype=torch.long)
        for r, s in enumerate(seqs):
            ids[r, L - len(s):] = torch.tensor(s)
            mask[r, L - len(s):] = 1
        return ids.to(DEVICE), mask.to(DEVICE), L

    @torch.inference_mode()
    def generate(self, seqs, max_new):
        ids, mask, L = self.pad_left(seqs)
        out = self.model.generate(input_ids=ids, attention_mask=mask, do_sample=False, num_beams=1,
                                  max_new_tokens=max_new, use_cache=True,
                                  pad_token_id=self.eos, eos_token_id=self.eos)
        res = []
        for row in out[:, L:].tolist():
            if self.eos in row:
                k, cap = row.index(self.eos), False
            else:
                k, cap = len(row), True
            res.append((self.tok.decode(row[:k]).strip(), k, cap))
        return res

    # ---- teacher-forced top-k on a target text (for token-level KD), image + direct prompt
    def tf_inputs(self, path, prompt, text):
        prefix = self.ids(path, prompt)
        conv = self.conv(path, prompt) + [{"from": "gpt", "value": text}]
        full = self.tok.apply_chat_template(conv, add_generation_prompt=False, return_tensors="pt")[0].tolist()
        tids = self.tok.encode(text, add_special_tokens=False)
        n = len(tids)
        if n == 0 or full[: len(prefix)] != prefix:
            return None
        start = None
        for s in range(len(prefix), len(full) - n + 1):
            if full[s: s + n] == tids:
                start = s
                break
        if start is None:
            return None
        return full[: start + n], start, tids

    @torch.inference_mode()
    def topk(self, items, k):
        """items: list of (path, prompt, text). Returns list of (ids[n,k] int32, logp[n,k] fp16, gt_logp_mean) or None."""
        prepped = [self.tf_inputs(*it) for it in items]
        ok = [p for p in prepped if p is not None]
        res = [None] * len(items)
        if not ok:
            return res
        ids, mask, L = self.pad_left([p[0] for p in ok])
        pos = (mask.cumsum(-1) - 1).clamp(min=0)
        try:
            out = self.model(input_ids=ids, attention_mask=mask, position_ids=pos, use_cache=False)
        except TypeError:
            out = self.model(input_ids=ids, attention_mask=mask, use_cache=False)
        logits = out.logits
        j = 0
        for r, p in enumerate(prepped):
            if p is None:
                continue
            seq, start, tids = p
            pad = L - len(seq)
            n = len(tids)
            lp = torch.log_softmax(logits[j, pad + start - 1: pad + start - 1 + n].float(), dim=-1)
            v, ix = lp.topk(k, dim=-1)
            gt = lp.gather(1, torch.tensor(tids, device=lp.device)[:, None])[:, 0]
            res[r] = (ix.int().cpu().numpy(), v.half().cpu().numpy(), float(gt.mean()))
            j += 1
        del logits, out
        return res

    # ---- one batch of the chosen route
    def run_route(self, idxs, route, use_ind, max_f, max_i, capture=True, ind_text=None):
        paths = [image_path(i) for i in idxs]

        def ind_for(i):
            if ind_text is not None:
                return ind_text
            if use_ind and IND_COL:
                return indication(i)
            return CFG.get("empty_indication_text", "")
        self.cap = {}
        self.capturing = capture
        try:
            if route == "two_step":
                seqs = [self.ids(p, PROMPT_FINDINGS.format(ind=ind_for(i))) for p, i in zip(paths, idxs)]
                fr = self.generate(seqs, max_f)
                self.capturing = False
                sseqs = [self.ids(None, PROMPT_SUMMARIZE.format(findings=f)) for f, _, _ in fr]
                ir = self.generate(sseqs, max_i)
            elif route == "direct":
                ir = self.generate([self.ids(p, PROMPT_DIRECT) for p in paths], max_i)
                fr = [("", 0, False)] * len(idxs)
            else:
                raise ValueError(route)
        finally:
            self.capturing = False
        cap = self.cap
        if capture:
            for key in ("siglip_mean", "siglip_grid4", "proj_mean"):
                assert key in cap and cap[key].shape[0] == len(idxs), f"feature capture failed for {key}"
            assert cap.get("n_encode_calls") == 1, f"visual.encode called {cap.get('n_encode_calls')} times (expected 1)"
        recs = []
        for b, i in enumerate(idxs):
            recs.append({"idx": int(i), "split": str(df.at[i, "subset"]), "route": route,
                         "indication_used": bool(ind_text is None and use_ind and IND_COL),
                         "findings": fr[b][0], "ntok_findings": fr[b][1], "cap_findings": fr[b][2],
                         "impression": ir[b][0], "ntok_impression": ir[b][1], "cap_impression": ir[b][2]})
        return recs, cap


# ------------------------------------------------------------------ text checks (probe only; rough, not CheXbert)
def words(t):
    return re.findall(r"[a-z0-9]+", t.lower())


def rouge_l(c, r, beta=1.2):
    c, r = words(c), words(r)
    if not c or not r:
        return 0.0
    dp = [0] * (len(r) + 1)
    for x in c:
        prev = 0
        for j, y in enumerate(r, 1):
            cur = dp[j]
            dp[j] = prev + 1 if x == y else max(dp[j], dp[j - 1])
            prev = cur
    lcs = dp[-1]
    if lcs == 0:
        return 0.0
    p, rc = lcs / len(c), lcs / len(r)
    return (1 + beta ** 2) * p * rc / (rc + beta ** 2 * p)


KW = {"effusion": r"effusion", "pneumothorax": r"pneumothora", "cardiomegaly": r"cardiomegaly|heart (size )?is enlarged|enlarged (heart|cardiac)",
      "edema": r"edema|vascular congestion", "consolidation": r"consolidat", "atelectasis": r"atelecta",
      "pneumonia": r"pneumonia", "opacity": r"opacit", "fracture": r"fracture", "lesion": r"nodule|mass\b|lesion",
      "devices": r"tube|line|catheter|pacemaker|wire|port\b|drain"}
NEG = re.compile(r"\b(no|without|negative for|free of|resolved|absence of|not)\b")


def kw_labels(t):
    out = {}
    for s in re.split(r"(?<=[.;])\s+|\n", t.lower()):
        for k, pat in KW.items():
            m = re.search(pat, s)
            if m:
                pos = not NEG.search(s[: m.start()])
                out[k] = out.get(k, False) or pos
    return {k: bool(out.get(k, False)) for k in KW}


def kw_agreement(preds, refs):
    tp = fp = fn = agree = tot = 0
    for p, r in zip(preds, refs):
        a, b = kw_labels(p), kw_labels(r)
        for k in KW:
            tot += 1
            agree += a[k] == b[k]
            tp += a[k] and b[k]
            fp += a[k] and not b[k]
            fn += b[k] and not a[k]
    f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    return {"label_accuracy": agree / max(tot, 1), "positive_f1": f1}


# ------------------------------------------------------------------ smoke (CPU, 2 images, a few tokens: checks mechanics, not quality)
def smoke():
    R = {"started": time.strftime("%Y-%m-%d %H:%M:%S"), "device": DEVICE, "dtype": str(DT)}
    T = Teacher()
    R.update(load_s=round(T.load_s, 1), dtype_census=T.dtype_census, vocab=T.vocab, eos=T.eos,
             tokenizer_special_ids={"img_start": T.tok.img_start_id, "img_end": T.tok.img_end_id, "img_pad": T.tok.img_pad_id})
    pool = df[(df["subset"] == "val") & df["use_for_training"]]
    sel = [int(i) for i in pool.index[:2]]
    n = CFG.get("smoke_new_tokens", 12)

    def timed_(fn):
        t = time.time(); out = fn(); return out, round(time.time() - t, 1)

    (recs, cap), t = timed_(lambda: T.run_route(sel, "direct", False, n, n))
    R["direct_batch2"] = {"s": t, "outputs": [r["impression"] for r in recs], "ntok": [r["ntok_impression"] for r in recs],
                          "feature_shapes": {k: list(v.shape) for k, v in cap.items() if hasattr(v, "shape")}, "siglip_tokens": cap.get("n_tokens"),
                          "finite": bool(all(np.isfinite(v.astype(np.float32)).all() for v in cap.values() if hasattr(v, "shape")))}
    assert R["direct_batch2"]["finite"], "non-finite visual features"
    (recs1, _), t = timed_(lambda: T.run_route(sel[:1], "direct", False, n, n))
    R["batch1_vs_batch2_same_first_output"] = bool(recs1[0]["impression"] == recs[0]["impression"])
    (recs2, cap2), t = timed_(lambda: T.run_route(sel[:1], "two_step", False, n, n))
    R["two_step_batch1"] = {"s": t, "findings": recs2[0]["findings"], "impression": recs2[0]["impression"]}
    tk, t = timed_(lambda: T.topk([(image_path(i), PROMPT_DIRECT, df.at[i, "impression_final"]) for i in sel], 5))
    R["topk"] = {"s": t, "aligned": f"{sum(x is not None for x in tk)}/{len(sel)}",
                 "shapes": [list(x[0].shape) if x is not None else None for x in tk]}
    R["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    json.dump(R, open(CFG["out_dir"] + "/smoke_report.json", "w"), indent=1)
    log("smoke done")


# ------------------------------------------------------------------ probe
def probe():
    R = {"started": time.strftime("%Y-%m-%d %H:%M:%S"), "indication_col": IND_COL, "device": DEVICE}
    if DEVICE.startswith("cuda"):
        R["gpu"] = torch.cuda.get_device_name(0)
    T = Teacher()
    R.update(load_s=round(T.load_s, 1), dtype_census=T.dtype_census, vocab=T.vocab)
    pool = df[(df["subset"] == "val") & df["use_for_training"]]
    rs = np.random.RandomState(0)
    nrm = pool[pool["y_no_finding"] == 1].index.values
    abn = pool[pool["y_no_finding"] != 1].index.values
    n = CFG.get("probe_n", 24)
    sel = list(rs.choice(nrm, min(n // 2, len(nrm)), replace=False)) + list(rs.choice(abn, min(n - n // 2, len(abn)), replace=False))
    sel = [int(i) for i in sel]
    gts = [df.at[i, "impression_final"] for i in sel]
    MF, MI = CFG["max_new_findings"], CFG["max_new_impression"]
    samples = {"idx": sel, "gt": gts, "normal": [int(df.at[i, "y_no_finding"] == 1) for i in sel]}

    def timed(fn):
        sync(); t = time.time(); out = fn(); sync()
        return out, time.time() - t

    def batched(idxs, bs, **kw):
        recs, feats = [], []
        for s in range(0, len(idxs), bs):
            r, c = T.run_route(idxs[s: s + bs], **kw)
            recs += r
            feats.append(c)
        return recs, feats

    # 0. warm-up + feature shapes
    (r0, c0), t0 = timed(lambda: T.run_route(sel[:2], "direct", False, MF, MI))
    R["warmup_s"] = round(t0, 1)
    R["features"] = {k: list(v.shape) for k, v in c0.items() if hasattr(v, "shape")}
    R["features"]["siglip_tokens"] = c0.get("n_tokens")
    R["features_finite"] = bool(all(np.isfinite(v.astype(np.float32)).all() for v in c0.values() if hasattr(v, "shape")))
    log(f"features {R['features']} finite={R['features_finite']}")

    # 1. batch-size equivalence (left padding must not change greedy outputs)
    eq = {}
    for route in ["direct", "two_step"]:
        sub = sel[:8]
        a, _ = batched(sub, 1, route=route, use_ind=bool(IND_COL), max_f=MF, max_i=MI)
        b, _ = batched(sub, 4, route=route, use_ind=bool(IND_COL), max_f=MF, max_i=MI)
        same = [x["impression"] == y["impression"] and x["findings"] == y["findings"] for x, y in zip(a, b)]
        eq[route] = {"identical_frac": float(np.mean(same)),
                     "rougeL_b1_vs_b4": float(np.mean([rouge_l(x["impression"] + " " + x["findings"], y["impression"] + " " + y["findings"]) for x, y in zip(a, b)]))}
        log(f"batch equivalence [{route}] {eq[route]}")
    R["batch_equivalence"] = eq

    # 2. prompt variants on the full probe set
    EMPTY = CFG.get("empty_indication_text", "")
    variants = [("direct", "direct", False, None),
                ("two_step_default_ind", "two_step", False, None)]          # indication text = EMPTY_INDICATION_TEXT
    if EMPTY != "None":
        variants.append(("two_step_ind_None", "two_step", False, "None"))
    if IND_COL:
        variants.append(("two_step_with_indication", "two_step", True, None))
    V = {}
    for name, route, use_ind, ind_text in variants:
        (recs, _), t = timed(lambda: batched(sel, 4, route=route, use_ind=use_ind, ind_text=ind_text, max_f=MF, max_i=MI))
        imps = [x["impression"] for x in recs]
        V[name] = {"s_per_img_b4": round(t / len(sel), 3),
                   "rougeL_vs_gt": round(float(np.mean([rouge_l(p, g) for p, g in zip(imps, gts)])), 4),
                   **{k: round(v, 4) for k, v in kw_agreement(imps, gts).items()},
                   "mean_tok_impression": float(np.mean([x["ntok_impression"] for x in recs])),
                   "cap_rate_impression": float(np.mean([x["cap_impression"] for x in recs])),
                   "mean_tok_findings": float(np.mean([x["ntok_findings"] for x in recs])),
                   "cap_rate_findings": float(np.mean([x["cap_findings"] for x in recs])),
                   "empty_impressions": int(sum(1 for p in imps if not p.strip()))}
        samples[name + "_impression"] = imps
        if route == "two_step":
            samples[name + "_findings"] = [x["findings"] for x in recs]
        log(f"variant {name}: {V[name]}")
    R["variants"] = V

    # 3. throughput vs batch size for each route (full route time per image)
    thr = {}
    for route in ["direct", "two_step"]:
        thr[route] = {}
        for bs in CFG.get("probe_batch_sizes", [1, 4, 8]):
            sub = (sel * 2)[: max(8, bs * 2)]
            try:
                _, t = timed(lambda: batched(sub, bs, route=route, use_ind=bool(IND_COL), max_f=MF, max_i=MI))
                thr[route][str(bs)] = round(t / len(sub), 3)
            except Exception as e:
                if not is_oom(e):
                    raise
                thr[route][str(bs)] = "OOM"
                torch.cuda.empty_cache()
            log(f"throughput {route} bs={bs}: {thr[route][str(bs)]} s/img")
            if DEVICE.startswith("cuda"):
                R.setdefault("peak_mem_gib", {})[f"{route}_bs{bs}"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
                torch.cuda.reset_peak_memory_stats()
    R["throughput_s_per_img"] = thr

    # 4. teacher-forced top-k logits on the GT impression (token-level KD)
    try:
        sub = sel[:8]
        items = [(image_path(i), PROMPT_DIRECT, df.at[i, "impression_final"]) for i in sub]
        single, t1 = timed(lambda: [T.topk([it], CFG["topk"])[0] for it in items])
        batch4, t4 = timed(lambda: T.topk(items[:4], CFG["topk"]) + T.topk(items[4:], CFG["topk"]))
        okn = sum(x is not None for x in single)
        top1, maxdiff = [], 0.0
        for a, b in zip(single, batch4):
            if a is None or b is None:
                continue
            top1.append(float((a[0][:, 0] == b[0][:, 0]).mean()))
            maxdiff = max(maxdiff, float(np.abs(a[1][:, 0].astype(np.float32) - b[1][:, 0].astype(np.float32)).max()))
        R["topk"] = {"aligned": f"{okn}/{len(sub)}", "s_per_img_b1": round(t1 / len(sub), 3), "s_per_img_b4": round(t4 / len(sub), 3),
                     "top1_agree_b1_vs_b4": float(np.mean(top1)) if top1 else None, "max_abs_logp_diff_b1_vs_b4": maxdiff,
                     "gt_mean_logp": float(np.mean([x[2] for x in single if x is not None])) if okn else None}
        log(f"topk: {R['topk']}")
    except Exception as e:
        R["topk"] = {"error": repr(e)[:500]}
        log(f"topk check failed: {e!r}")

    R["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    json.dump(R, open(CFG["out_dir"] + "/probe_report.json", "w"), indent=2)
    pd.DataFrame(samples).to_csv(CFG["out_dir"] + "/probe_samples.csv", index=False)
    log("probe done")


# ------------------------------------------------------------------ cache
def save_chunk(recs, feats, tk):
    if not recs:
        return
    name = f"g{RANK}_{int(time.time() * 1000)}_{len(recs)}"
    d = CFG["cache_dir"]
    arrays = {"idx": np.array([r["idx"] for r in recs], dtype=np.int64)}
    if CFG["cache_features"]:
        for key in ("siglip_mean", "siglip_grid4", "proj_mean"):
            arrays[key] = np.concatenate([f[key] for f in feats], 0)
            assert arrays[key].shape[0] == len(recs)
    if CFG["cache_topk"]:
        ids = [t[0] if t is not None else np.zeros((0, CFG["topk"]), np.int32) for t in tk]
        lps = [t[1] if t is not None else np.zeros((0, CFG["topk"]), np.float16) for t in tk]
        arrays["topk_ids"] = np.concatenate(ids, 0)
        arrays["topk_logp"] = np.concatenate(lps, 0)
        arrays["topk_offsets"] = np.cumsum([0] + [len(x) for x in ids]).astype(np.int64)
    tmp = f"{d}/{name}.tmp.npz"
    np.savez(tmp, **arrays)
    os.replace(tmp, f"{d}/{name}.npz")                      # npz lands atomically BEFORE its jsonl lines
    with open(f"{d}/g{RANK}.jsonl", "a") as fh:
        for r, t in zip(recs, tk if CFG["cache_topk"] else [None] * len(recs)):
            r = dict(r, chunk=name)
            if CFG["cache_topk"]:
                r["topk_ok"] = t is not None
                r["teacher_gt_logp_mean"] = t[2] if t is not None else None
            fh.write(json.dumps(r) + "\n")
        fh.flush()
        os.fsync(fh.fileno())


def cache():
    ids = json.load(open(args.ids))
    deadline = CFG["deadline"]
    T = Teacher()
    os.makedirs(CFG["cache_dir"], exist_ok=True)
    route, use_ind = CFG["route"], CFG["use_indication"]
    MF, MI, BS, CH = CFG["max_new_findings"], CFG["max_new_impression"], CFG["batch_size"], CFG["chunk"]
    buf_r, buf_f, buf_t = [], [], []
    done, failed, t_start, batch_times = 0, 0, time.time(), []
    fail_fh = open(f"{CFG['cache_dir']}/g{RANK}_failed.jsonl", "a")

    def run(b):
        """Run one batch; on OOM split in half; on other errors fall back to singles. Returns list of (rec, feat_rows, topk)."""
        try:
            recs, cap = T.run_route(b, route, use_ind, MF, MI, capture=True)
            tk = [None] * len(b)
            if CFG["cache_topk"]:
                tk = T.topk([(image_path(i), PROMPT_DIRECT, df.at[i, "impression_final"]) for i in b], CFG["topk"])
            out = []
            for j, r in enumerate(recs):
                f = {k: cap[k][j: j + 1] for k in ("siglip_mean", "siglip_grid4", "proj_mean")} if CFG["cache_features"] else None
                out.append((r, f, tk[j]))
            return out
        except Exception as e:
            oom = is_oom(e)
            if DEVICE.startswith("cuda"):
                torch.cuda.empty_cache()
            if len(b) > 1:
                log(f"{'OOM' if oom else 'error'} on batch of {len(b)} ({repr(e)[:200]}); splitting")
                h = len(b) // 2 if oom else 1
                res = []
                for s in range(0, len(b), h):
                    res += run(b[s: s + h])
                return res
            log(f"FAILED idx {b[0]}: {repr(e)[:300]}")
            fail_fh.write(json.dumps({"idx": int(b[0]), "error": repr(e)[:1000], "tb": traceback.format_exc()[-2000:]}) + "\n")
            fail_fh.flush()
            return []

    stopped_early = False
    for s in range(0, len(ids), BS):
        est = np.median(batch_times[-20:]) if batch_times else 60.0
        if time.time() + 2 * est + 120 > deadline:
            log(f"deadline guard: stopping with {len(ids) - s} images left for the next session")
            stopped_early = True
            break
        b = ids[s: s + BS]
        sync(); t = time.time()
        res = run(b)
        sync(); batch_times.append(time.time() - t)
        failed += len(b) - len(res)
        for r, f, tk in res:
            buf_r.append(r); buf_f.append(f); buf_t.append(tk)
        for i in b:
            p = f"{TMP}/{i}.png"
            if os.path.exists(p):
                os.remove(p)
        if len(buf_r) >= CH:
            save_chunk(buf_r, buf_f, buf_t)
            done += len(buf_r)
            buf_r, buf_f, buf_t = [], [], []
            el = time.time() - t_start
            rate = el / max(done, 1)
            log(f"saved {done}/{len(ids)} | {rate:.2f} s/img on this GPU | ETA {(len(ids) - done) * rate / 3600:.2f} h | failed {failed}")
    save_chunk(buf_r, buf_f, buf_t)
    done += len(buf_r)
    log(f"worker finished: saved {done}, failed {failed}, stopped_early={stopped_early}")
    json.dump({"rank": RANK, "saved": done, "failed": failed, "stopped_early": stopped_early,
               "s_per_img": (time.time() - t_start) / max(done, 1)},
              open(f"{CFG['cache_dir']}/g{RANK}_last_run.json", "w"))


if __name__ == "__main__":
    try:
        {"smoke": smoke, "probe": probe, "cache": cache}[args.mode]()
    except Exception:
        log("FATAL\n" + traceback.format_exc())
        sys.exit(1)

In [ ]:
WORKER = f"{WORK}/teacher_worker.py"

def write_cfg(extra):
    cfg = dict(csv=CSV, npy=NPY, indication_col=IND_COL, device=DEVICE, tmp_dir=TMP_DIR, out_dir=WORK, cache_dir=CACHE,
               route=ROUTE, use_indication=bool(USE_INDICATION and IND_COL), cache_features=CACHE_FEATURES,
               cache_topk=CACHE_TOPK, topk=TOPK, batch_size=BATCH_SIZE, chunk=CHUNK, empty_indication_text=EMPTY_INDICATION_TEXT,
               dtype="float32" if DEVICE == "cpu" else "float16", smoke_new_tokens=12,
               max_new_findings=MAX_NEW_FINDINGS, max_new_impression=MAX_NEW_IMPRESSION,
               deadline=T0 + SESSION_BUDGET_H * 3600, probe_n=PROBE_N, probe_batch_sizes=[1, 4, 8])
    cfg.update(extra)
    p = f"{WORK}/worker_cfg.json"
    json.dump(cfg, open(p, "w"), indent=1)
    return p

def tail(path, n=3):
    try:
        return open(path).read().splitlines()[-n:]
    except FileNotFoundError:
        return []

def launch(rank, mode, cfg_path, ids_path=None):
    stamp = time.strftime("%Y%m%d_%H%M%S")
    log = f"{LOGS}/{mode}_gpu{rank}_{stamp}.log"
    env = dict(os.environ, MPLBACKEND="agg", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    if DEVICE.startswith("cuda"):
        env["CUDA_VISIBLE_DEVICES"] = str(rank)
    cmd = [PY, WORKER, "--rank", str(rank), "--mode", mode, "--cfg", cfg_path] + (["--ids", ids_path] if ids_path else [])
    p = subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT, env=env)
    print(f"launched {mode} worker gpu{rank} (pid {p.pid}) -> {log}")
    return p, log

def wait_until_loaded(p, log, timeout=45 * 60):
    """Start the next worker only after this one has loaded: avoids two simultaneous 12.6 GB downloads and two CPU-side loads."""
    t = time.time()
    while time.time() - t < timeout:
        txt = "\n".join(tail(log, 50))
        if "model loaded" in txt or p.poll() is not None:
            return
        time.sleep(15)

def monitor(jobs, every_min=10):
    last = 0
    while any(p.poll() is None for p, _ in jobs):
        time.sleep(30)
        if time.time() - last > every_min * 60:
            last = time.time()
            for p, log in jobs:
                print(f"--- {os.path.basename(log)} ({'running' if p.poll() is None else 'exit ' + str(p.returncode)})")
                for l in tail(log, 2):
                    print("   ", l)
        if time.time() > T0 + SESSION_BUDGET_H * 3600 + 20 * 60:      # hard stop; workers should have stopped already
            for p, _ in jobs:
                if p.poll() is None:
                    p.terminate()
            print("hard stop: terminated workers past the session budget")
    return [p.returncode for p, _ in jobs]

In [ ]:
if MODE == "smoke":
    scfg = write_cfg({})
    p, log = launch(0, "smoke", scfg)
    rc = monitor([(p, log)], every_min=2)
    print("\n".join(tail(log, 60)))
    assert rc[0] == 0, "smoke test failed - see the log above (no GPU quota was used)"
    S = json.load(open(f"{WORK}/smoke_report.json"))
    print(json.dumps(S, indent=1))
    print("\nSMOKE TEST PASSED: the real model loaded and ran through every code path the probe/cache use. Now switch to the GPU and set MODE = 'probe'.")

In [ ]:
if MODE == "probe":
    pcfg = write_cfg({})
    p, log = launch(0, "probe", pcfg)
    rc = monitor([(p, log)], every_min=3)
    print("\n".join(tail(log, 40)))
    assert rc[0] == 0, "probe failed - see the log above"
    R = json.load(open(f"{WORK}/probe_report.json"))
    print(json.dumps({k: v for k, v in R.items() if k not in ("variants", "throughput_s_per_img")}, indent=1))
    print("\nVARIANTS (24 val images, batch 4; label scores are a rough keyword check, not CheXbert):")
    print(pd.DataFrame(R["variants"]).T.to_string())
    print("\nTHROUGHPUT, seconds per image for the whole route:")
    print(pd.DataFrame(R["throughput_s_per_img"]).to_string())
    n_todo = int((df.use_for_training & df.subset.isin(SPLITS)).sum()) - len([r for r in done.values() if r["split"] in SPLITS])
    print(f"\nPROJECTED GPU time for {n_todo} images on {N_GPU} GPU(s) (wall-clock; excludes ~5 min load per session):")
    for route, row in R["throughput_s_per_img"].items():
        for bs, s in row.items():
            if isinstance(s, (int, float)):
                extra = R.get("topk", {}).get("s_per_img_b4") or 0
                print(f"  {route:9s} bs={bs}: {n_todo * s / N_GPU / 3600:5.1f} h   (+ top-k logits: {n_todo * (s + extra) / N_GPU / 3600:5.1f} h)")
    S = pd.read_csv(f"{WORK}/probe_samples.csv")
    pd.set_option("display.max_colwidth", 220)
    print("\nSAMPLE OUTPUTS (first 6):")
    for _, r in S.head(6).iterrows():
        print(f"\nidx {r['idx']} | normal={r['normal']}\n  GT      : {r['gt']}")
        for c in S.columns:
            if c.endswith("_impression"):
                print(f"  {c[:-11]:24s}: {r[c]}")

In [ ]:
if MODE == "cache":
    routes = {r["route"] for r in done.values()}
    assert routes <= {ROUTE}, f"cache already holds route(s) {routes}; refusing to mix with ROUTE={ROUTE!r}"
    if done:
        prev_cfg = json.load(open(f"{CACHE}/run_config.json")) if os.path.exists(f"{CACHE}/run_config.json") else {}
        for k, v in dict(cache_features=CACHE_FEATURES, cache_topk=CACHE_TOPK, topk=TOPK, use_indication=bool(USE_INDICATION and IND_COL), empty_indication_text=EMPTY_INDICATION_TEXT,
                         max_new_findings=MAX_NEW_FINDINGS, max_new_impression=MAX_NEW_IMPRESSION).items():
            assert prev_cfg.get(k, v) == v, f"{k} was {prev_cfg.get(k)} in earlier sessions, now {v}; keep it the same"
    json.dump(dict(route=ROUTE, cache_features=CACHE_FEATURES, cache_topk=CACHE_TOPK, topk=TOPK,
                   use_indication=bool(USE_INDICATION and IND_COL), indication_col=IND_COL, empty_indication_text=EMPTY_INDICATION_TEXT,
                   max_new_findings=MAX_NEW_FINDINGS, max_new_impression=MAX_NEW_IMPRESSION),
              open(f"{CACHE}/run_config.json", "w"), indent=1)

    order = {s: k for k, s in enumerate(SPLITS)}
    pool = df[df.use_for_training & df.subset.isin(SPLITS)].copy()
    pool["o"] = pool.subset.map(order)
    todo = [int(i) for i in pool.sort_values(["o", "idx"]).idx if int(i) not in done]
    print(f"to do: {len(todo)} of {len(pool)}")
    if MAX_IMAGES_THIS_SESSION:
        todo = todo[:MAX_IMAGES_THIS_SESSION]
    if todo:
        cfg = write_cfg({})
        jobs = []
        for r in range(N_GPU):
            ids_r = todo[r::N_GPU]
            idp = f"{WORK}/ids_gpu{r}.json"
            json.dump(ids_r, open(idp, "w"))
            p, log = launch(r, "cache", cfg, idp)
            jobs.append((p, log))
            if r < N_GPU - 1:
                wait_until_loaded(p, log)
        rcs = monitor(jobs)
        # one relaunch for a worker that crashed, if there is time left
        for r, ((p, log), rc) in enumerate(zip(jobs, rcs)):
            if rc != 0:
                print(f"worker gpu{r} exited with {rc}; log tail:"); print("\n".join(tail(log, 25)))
                now_done = load_done()
                left = [i for i in todo[r::N_GPU] if i not in now_done]
                if left and time.time() < T0 + SESSION_BUDGET_H * 3600 - 45 * 60:
                    idp = f"{WORK}/ids_gpu{r}_retry.json"; json.dump(left, open(idp, "w"))
                    p2, log2 = launch(r, "cache", cfg, idp)
                    monitor([(p2, log2)])
    done = load_done()
    allpool = df[df.use_for_training & df.subset.isin(SPLITS)]
    rem = [int(i) for i in allpool.idx if int(i) not in done]
    print("\n================ SESSION SUMMARY ================")
    print(f"cached: {len(done)} | remaining for {SPLITS}: {len(rem)}")
    print("by split:", pd.Series([r["split"] for r in done.values()]).value_counts().to_dict())
    fails = [json.loads(l) for f in glob.glob(f"{CACHE}/*_failed.jsonl") for l in open(f) if l.strip()]
    print(f"failed attempts logged: {len(fails)} (failed images are retried next session)")
    for f in sorted(glob.glob(f"{CACHE}/g*_last_run.json")):
        print(" ", json.load(open(f)))
    recs = list(done.values())
    if recs:
        print(f"impression cap-hit rate {np.mean([r['cap_impression'] for r in recs]):.2%} | empty impressions {sum(not r['impression'].strip() for r in recs)}")
        if ROUTE == "two_step":
            print(f"findings cap-hit rate {np.mean([r['cap_findings'] for r in recs]):.2%}")
    print(f"session wall time {(time.time() - T0) / 3600:.2f} h")
    for f in glob.glob(f"{TMP_DIR}/cxr_g*"):
        shutil.rmtree(f, ignore_errors=True)

In [ ]:
if MODE == "merge":
    done = load_done()
    OUT = f"{WORK}/teacher_final"
    os.makedirs(OUT, exist_ok=True)
    idxs = sorted(done)
    by_chunk = {}
    for i in idxs:
        by_chunk.setdefault(done[i]["chunk"], []).append(i)
    pos = {i: k for k, i in enumerate(idxs)}
    feats, tk_ids, tk_lp, tk_len = {}, [None] * len(idxs), [None] * len(idxs), np.zeros(len(idxs), np.int64)
    for ch, members in by_chunk.items():
        z = np.load(f"{CACHE}/{ch}.npz")
        row = {int(v): k for k, v in enumerate(z["idx"])}
        for key in ("siglip_mean", "siglip_grid4", "proj_mean"):
            if key in z:
                if key not in feats:
                    feats[key] = np.zeros((len(idxs),) + z[key].shape[1:], np.float16)
                for i in members:
                    feats[key][pos[i]] = z[key][row[i]]
        if "topk_ids" in z:
            off = z["topk_offsets"]
            for i in members:
                a, b = off[row[i]], off[row[i] + 1]
                tk_ids[pos[i]], tk_lp[pos[i]], tk_len[pos[i]] = z["topk_ids"][a:b], z["topk_logp"][a:b], b - a
    with open(f"{OUT}/teacher_cache.jsonl", "w") as fh:
        for i in idxs:
            fh.write(json.dumps(done[i]) + "\n")
    if feats:
        np.savez(f"{OUT}/teacher_feats.npz", idx=np.array(idxs), **feats)
    if any(x is not None for x in tk_ids):
        K = next(x.shape[1] for x in tk_ids if x is not None)
        tk_ids = [x if x is not None else np.zeros((0, K), np.int32) for x in tk_ids]
        tk_lp = [x if x is not None else np.zeros((0, K), np.float16) for x in tk_lp]
        np.savez(f"{OUT}/teacher_topk.npz", idx=np.array(idxs), offsets=np.concatenate([[0], np.cumsum(tk_len)]),
                 ids=np.concatenate(tk_ids), logp=np.concatenate(tk_lp))
    pool = df[df.use_for_training & df.subset.isin(SPLITS)]
    miss = sorted(set(pool.idx.astype(int)) - set(idxs))
    rep = {"cached": len(idxs), "missing_from_SPLITS": len(miss), "missing_idx_first20": miss[:20],
           "by_split": pd.Series([done[i]["split"] for i in idxs]).value_counts().to_dict(),
           "features": {k: list(v.shape) for k, v in feats.items()},
           "features_finite": bool(all(np.isfinite(v.astype(np.float32)).all() for v in feats.values())),
           "empty_impressions": int(sum(not done[i]["impression"].strip() for i in idxs)),
           "impression_cap_rate": float(np.mean([done[i]["cap_impression"] for i in idxs])) if idxs else None}
    if os.path.exists(f"{CACHE}/run_config.json"):
        rep["run_config"] = json.load(open(f"{CACHE}/run_config.json"))
    json.dump(rep, open(f"{OUT}/merge_report.json", "w"), indent=1)
    print(json.dumps(rep, indent=1))
    print("files:", {f: f"{os.path.getsize(f'{OUT}/{f}') / 2**20:.1f} MB" for f in os.listdir(OUT)})